# 🤖 Generative AI — Lecture 2
## Derivation of Generative Models
### MAP · MLE · Beta-Binom · Dirichlet

**Haydar Kılıç | Faculty of Engineering, Artificial Intelligence Engineering**

---
This notebook contains hands-on Python demonstrations of the theoretical concepts covered in the lecture slides.

In [ ]:
# [H5] comb removed — not used
# [H6] dirichlet (scipy.stats) removed — using np.random.dirichlet
# [H7] mpatches removed — not used anywhere
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import beta as beta_dist
from scipy.stats import binom
from scipy.special import betaln, gammaln
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# [H1] np.trapz removed in NumPy 2.0 → use np.trapezoid
# Backward-compatibility wrapper:
if not hasattr(np, 'trapezoid'):
    np.trapezoid = np.trapz  # NumPy < 2.0 support

plt.rcParams['figure.figsize'] = (11, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

NAVY = '#1a237e'
BLUE = '#1565c0'
print('✅ Libraries loaded.')

---
## 📌 SECTION 1: Learning from Positive Examples & The Number Game

### 1.1 Concept Learning = Binary Classification

Bir çocuğun kavramları **yalnızca pozitif örneklerden** öğrendiği gibi, modelimiz de  
D = {x₁, …, xₙ} pozitif örneklerinden hangi sayıların C kümesine ait olduğunu öğrenir.

**Posterior Predictive Distribution:** $p(\tilde{x} \in C \mid \mathcal{D})$ — the probability that a new number falls within C.

In [ ]:
# ── Number Game: Defining the Hypothesis Space ─────────────────────────────────
UNIVERSE = set(range(1, 101))   # {1, …, 100}

def build_hypotheses():
    """Arithmetic hypotheses defined over 1-100."""
    H = {}
    H['odd numbers']   = {x for x in UNIVERSE if x % 2 != 0}
    H['even numbers']  = {x for x in UNIVERSE if x % 2 == 0}
    H['squares']       = {x for x in UNIVERSE if int(x**0.5)**2 == x}
    for k in range(3, 11):
        H[f'multiples of {k}'] = {x for x in UNIVERSE if x % k == 0}
    for j in range(10):
        H[f'ending in {j}']    = {x for x in UNIVERSE if x % 10 == j}
    for b in range(2, 11):
        H[f'powers of {b}'] = set()
        v = b
        while v <= 100:
            H[f'powers of {b}'].add(v)
            v *= b
    H['all'] = set(UNIVERSE)
    # Intervals (increments of 10)
    for lo in range(1, 100, 10):
        hi = min(lo + 9, 100)
        H[f'interval [{lo},{hi}]'] = {x for x in UNIVERSE if lo <= x <= hi}
    return H

hypotheses = build_hypotheses()

print(f'📐 Total number of hypotheses: {len(hypotheses)}')
print('\nSome hypotheses:')
for name in ['powers of 2', 'even numbers', 'squares', 'odd numbers']:
    s = sorted(hypotheses[name])
    print(f'  {name:25s}: {s[:10]} … |h|={len(s)}')

---
### 1.2 Strong Sampling Assumption & Size Principle

Assuming N positive examples are sampled **uniformly** from hypothesis h:

$$p(\mathcal{D} \mid h) = \left(\frac{1}{|h|}\right)^N$$

**Size Principle:** A narrower hypothesis (small |h|) → higher likelihood.

In [ ]:
# ── Numerical Demonstration of the Size Principle ───────────────────────────────
def likelihood(data, h_set):
    """p(D|h) = (1/|h|)^N  if D ⊆ h, else 0."""
    if not set(data).issubset(h_set):
        return 0.0
    return (1.0 / len(h_set)) ** len(data)

# Example: compare two competing hypotheses for D = {8}
D_example = [8]
h1 = hypotheses['powers of 2']    # narrow: {2,4,8,16,32,64} -> |h|=6
h2 = hypotheses['even numbers']   # broad: 50 elements

L1 = likelihood(D_example, h1)
L2 = likelihood(D_example, h2)

print('🎯 SIZE PRINCIPLE — Likelihood Comparison for D = {8}')
print('=' * 58)
print(f"h₁ = 'powers of 2'    |h₁| = {len(h1):3d}  p(D|h₁) = {L1:.6f}")
print(f"h₂ = 'even numbers'   |h₂| = {len(h2):3d}  p(D|h₂) = {L2:.6f}")
print(f"\n  Likelihood ratio L1/L2 = {L1/L2:.2f}x")
print(f"  → h₁ explains the data {L1/L2:.1f}x better than h₂!")

# How does the gap widen as N increases?
fig, ax = plt.subplots(figsize=(10, 5))
N_vals = np.arange(1, 10)
ratio  = [(len(h2)/len(h1))**n for n in N_vals]
ax.semilogy(N_vals, ratio, 'bo-', lw=2.5, ms=8)
ax.set_xlabel('Number of Observations N', fontsize=13)
ax.set_ylabel('Likelihood Ratio p(D|h₁) / p(D|h₂)  [log scale]', fontsize=12)
ax.set_title('Size Principle: Narrow hypothesis dominates as N grows', fontsize=13, fontweight='bold')
ax.set_xticks(N_vals)
for n, r in zip(N_vals, ratio):
    ax.annotate(f'{r:.0f}x', (n, r), textcoords='offset points', xytext=(4, 4), fontsize=9)
plt.tight_layout()
plt.show()

---
### 1.3 Prior, Likelihood, Posterior — Bayesian Update

$$p(h \mid \mathcal{D}) = \frac{p(\mathcal{D}\mid h)\, p(h)}{\sum_{h'} p(\mathcal{D}\mid h')\, p(h')}$$

In [ ]:
# ── Prior Definition ──────────────────────────────────────────────────────────────
def build_prior(hypotheses, pi0=0.6):
    """Mixture prior: π₀·p_rules + (1-π₀)·p_interval."""
    rule_keys     = [k for k in hypotheses if 'interval' not in k]
    interval_keys = [k for k in hypotheses if 'interval' in  k]
    prior = {}
    for k in rule_keys:
        prior[k] = pi0 / len(rule_keys)
    for k in interval_keys:
        prior[k] = (1 - pi0) / len(interval_keys)
    return prior

def compute_posterior(data, hypotheses, prior):
    """Compute posterior probabilities via Bayes' rule."""
    unnorm = {}
    for h, h_set in hypotheses.items():
        unnorm[h] = likelihood(data, h_set) * prior.get(h, 0.0)
    Z = sum(unnorm.values())
    if Z == 0:
        return {h: 0.0 for h in hypotheses}
    return {h: v / Z for h, v in unnorm.items()}

def posterior_predictive(x_tilde, posterior, hypotheses):
    """BMA: p(x̃ ∈ C | D) = Σ_h p(y=1|x̃,h)·p(h|D)"""
    prob = 0.0
    for h, p_h in posterior.items():
        prob += (1.0 if x_tilde in hypotheses[h] else 0.0) * p_h
    return prob

# ── Visualisation: D={16} and D={16,8,2,64} ──────────────────────────────────
prior = build_prior(hypotheses, pi0=0.7)

scenarios = [
    {'data': [16],          'title': 'D = {16}',
     'color': '#42A5F5'},
    {'data': [16, 8, 2, 64],'title': 'D = {16, 8, 2, 64}',
     'color': '#1565c0'},
    {'data': [16,23,19,20], 'title': 'D = {16, 23, 19, 20}',
     'color': '#7B1FA2'},
]

fig, axes = plt.subplots(len(scenarios), 1, figsize=(13, 11))
fig.suptitle('Number Game — Posterior Predictive Distribution p(x̃ ∈ C | D)',
             fontsize=14, fontweight='bold')

xs = np.arange(1, 101)
for ax, sc in zip(axes, scenarios):
    post = compute_posterior(sc['data'], hypotheses, prior)
    preds = np.array([posterior_predictive(x, post, hypotheses) for x in xs])
    preds /= (preds.max() + 1e-12)   # normalize to [0,1] for display

    ax.bar(xs, preds, color=sc['color'], alpha=0.85, width=0.8)
    for d in sc['data']:
        ax.axvline(x=d, color='red', lw=1.5, alpha=0.7)
    ax.set_title(sc['title'] + '  → positive examples marked with red lines',
                 fontsize=12, fontweight='bold')
    ax.set_xlim(0, 101)
    ax.set_ylim(0, 1.15)
    ax.set_xticks(range(0, 101, 4))
    ax.set_ylabel('p(x̃ ∈ C|D)', fontsize=10)

axes[-1].set_xlabel('x̃', fontsize=12)
plt.tight_layout()
plt.show()

---
### 1.4 MAP Estimation and N → ∞ Behaviour

With sufficient data the posterior concentrates on a single hypothesis (Dirac measure):

$$p(h \mid \mathcal{D}) \to \delta_{\hat{h}_{MAP}}(h), \qquad \hat{h}_{MAP} = \arg\max_h\, p(h\mid\mathcal{D})$$

**MLE:** When data is abundant the prior is overwhelmed → MAP → MLE

In [ ]:
# ── MAP sharpening as N grows ──────────────────────────────────────────────────
true_concept = hypotheses['powers of 2']   # true hidden concept
true_samples = sorted(true_concept)         # numbers that can be sampled

np.random.seed(42)
sample_sizes = [1, 2, 4, 8]

# Visualisation: posterior probability of top-8 hypotheses vs N
key_hyps = ['powers of 2', 'even numbers', 'powers of 4',
            'squares', 'multiples of 4', 'multiples of 8', 'all', 'odd numbers']
colors_k  = ['#D32F2F','#1565c0','#F57C00','#388E3C',
              '#7B1FA2','#00838F','#455A64','#BF360C']

fig, axes = plt.subplots(1, len(sample_sizes), figsize=(16, 5), sharey=True)
fig.suptitle('MAP Yakınsaması: N arttıkça Sonsal Keskinleşir\n(Gerçek kavram: "2\'nin kuvvetleri")',
             fontsize=13, fontweight='bold')

for ax, N in zip(axes, sample_sizes):
    data = list(np.random.choice(true_samples, size=N, replace=True))
    post = compute_posterior(data, hypotheses, prior)

    vals  = [post.get(h, 0) for h in key_hyps]
    bars  = ax.barh(range(len(key_hyps)), vals,
                    color=colors_k, alpha=0.85, edgecolor='white')
    ax.set_yticks(range(len(key_hyps)))
    # [H3] DÜZELTİLDİ: ' nin ' → "'nin " (etiket bozulması giderildi)
    ax.set_yticklabels([h.replace(' nin ', "'nin ") for h in key_hyps], fontsize=9)
    ax.set_title(f'N = {N}\nD = {sorted(set(data))}', fontsize=10)
    ax.set_xlabel('p(h|D)')
    ax.set_xlim(0, 1.0)

plt.tight_layout()
plt.show()

# MAP vs MLE comparison (textual summary)
# [H2] FIXED: f-string outer delimiter single quote → double quote
print('\n📊 MAP vs MLE Comparison:')
print('─'*52)
print(f"  {'N':>4}  {'MAP hypothesis':30}  {'p(h|D)'}")
print('─'*52)
for N in [1, 2, 4, 8, 16]:
    data = list(np.random.choice(true_samples, size=N, replace=True))
    post = compute_posterior(data, hypotheses, prior)
    map_h  = max(post, key=post.get)
    map_p  = post[map_h]
    print(f'  {N:>4}  {map_h:30}  {map_p:.4f}')

---
### 1.5 Bayesian Model Averaging (BMA) vs Plug-In Approximation

| Method | Formula | Advantage | Disadvantage |
|--------|--------|---------|------------|
| **BMA** | $p(\tilde{x}\|D)=\sum_h p(\tilde{x}\|h)p(h\|D)$ | Preserves uncertainty | Computationally expensive |
| **Plug-In** | $p(\tilde{x}\|D)=p(\tilde{x}\|\hat{h})$ | Fast and simple | Uncertainty is lost |

In [ ]:
# ── BMA vs Plug-In ──────────────────────────────────────────────────────────────
data_bma = [16, 8, 2, 64]
post_bma  = compute_posterior(data_bma, hypotheses, prior)
map_h_bma = max(post_bma, key=post_bma.get)

# BMA estimate
bma_preds    = np.array([posterior_predictive(x, post_bma, hypotheses) for x in xs])

# Plug-in estimate (use only the MAP hypothesis)
plugin_preds = np.array([1.0 if x in hypotheses[map_h_bma] else 0.0 for x in xs])

fig, axes = plt.subplots(2, 1, figsize=(13, 8))
fig.suptitle(f'BMA vs Plug-In Approximation  (D = {data_bma})', fontsize=14, fontweight='bold')

for ax, preds, title, color in zip(
        axes,
        [bma_preds / bma_preds.max(), plugin_preds],
        [f'BMA — Weighted average over all hypotheses',
         f'Tak-Çıkar — Yalnızca MAP: "{map_h_bma}" (p={post_bma[map_h_bma]:.3f})'],
        ['#1565c0', '#C62828']):
    ax.bar(xs, preds, color=color, alpha=0.8, width=0.8)
    for d in data_bma:
        ax.axvline(x=d, color='gold', lw=2, alpha=0.9, label='Training data' if d == data_bma[0] else '')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlim(0, 101); ax.set_ylim(0, 1.2)
    ax.set_xticks(range(0, 101, 4))
    ax.set_ylabel('p(x̃ ∈ C|D)')
    ax.legend()

axes[-1].set_xlabel('x̃')
plt.tight_layout()
plt.show()

print('💡 BMA spreads uncertainty across the distribution, while Plug-In concentrates')
print(f'   MAP hipotezi "{map_h_bma}" üzerine yığar.')

---
## 📌 SECTION 2: Parametric Bayes — Beta-Binomial Model
### 2.1 Bernoulli Likelihood and Sufficient Statistics

$X_i \sim \text{Ber}(\theta)$ için:

$$p(\mathcal{D}\mid\theta) = \theta^{N_1}(1-\theta)^{N_0}$$

$(N_1, N_0)$ — **yeterli istatistikler**: θ hakkında bilmemiz gereken tek şey.

In [ ]:
# ── Bernoulli Likelihood Surface ──────────────────────────────────────────────
theta_vals = np.linspace(0.001, 0.999, 500)

scenarios_bern = [
    {'N1': 3, 'N0': 0, 'label': 'N₁=3, N₀=0 (3 heads)', 'color': '#C62828'},
    {'N1': 7, 'N0': 3, 'label': 'N₁=7, N₀=3 (7H 3T)',  'color': '#1565c0'},
    {'N1': 5, 'N0': 5, 'label': 'N₁=5, N₀=5 (balanced)', 'color': '#2E7D32'},
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Bernoulli Likelihood Function  p(D|θ) = θᴺ¹(1-θ)ᴺ⁰',
             fontsize=14, fontweight='bold')

for ax, sc in zip(axes, scenarios_bern):
    L = theta_vals ** sc['N1'] * (1 - theta_vals) ** sc['N0']
    L /= L.max()    # normalize
    mle = sc['N1'] / (sc['N1'] + sc['N0'])

    ax.plot(theta_vals, L, color=sc['color'], lw=2.5)
    ax.axvline(x=mle, color='black', ls='--', lw=2,
               label=f'MLE θ̂ = {mle:.2f}')
    ax.fill_between(theta_vals, L, alpha=0.15, color=sc['color'])
    ax.set_title(sc['label'], fontsize=11)
    ax.set_xlabel('θ'); ax.set_ylabel('p(D|θ)  [normalize]')
    ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

print('📌 Sufficient Statistic Summary:')
print('   All information is stored in the pair (N₁, N₀)!')
print('   Order of observations does not matter: HHTHHT ≡ HHHHTT etc.')

---
### 2.2 Beta Distribution — Conjugate Prior

$$\text{Beta}(\theta \mid a, b) \propto \theta^{a-1}(1-\theta)^{b-1}$$

- **a** = pseudo-head count, **b** = pseudo-tail count (hyperparameters)
- Conjugacy: Beta prior × Bernoulli likelihood → **Beta posterior**
- $\mathbb{E}[\theta] = \dfrac{a}{a+b}$

In [ ]:
# ── Beta Distribution: Different (a,b) parameters ─────────────────────────────────
theta_plot = np.linspace(0.001, 0.999, 500)

params = [
    (1,  1,  '#78909C', 'Beta(1,1) — Uniform prior'),
    (2,  2,  '#42A5F5', 'Beta(2,2) — Mild central tendency'),
    (5,  2,  '#EF5350', 'Beta(5,2) — Heads bias (a>b)'),
    (2,  5,  '#66BB6A', 'Beta(2,5) — Tails bias (b>a)'),
    (10, 10, '#AB47BC', 'Beta(10,10) — Strong central prior'),
    (0.5,0.5,'#FF7043', 'Beta(0.5,0.5) — U-shaped (Jeffreys)'),
]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle('Beta Distribution: Different Hyperparameters (a, b)', fontsize=14, fontweight='bold')

for ax, (a, b, color, label) in zip(axes.flat, params):
    pdf = beta_dist.pdf(theta_plot, a, b)
    mean = a / (a + b)
    mode = (a - 1) / (a + b - 2) if (a > 1 and b > 1) else None

    ax.plot(theta_plot, pdf, color=color, lw=2.5)
    ax.fill_between(theta_plot, pdf, alpha=0.2, color=color)
    ax.axvline(x=mean, color='blue',  ls='--', lw=1.5, label=f'Mean = {mean:.2f}')
    if mode is not None:
        ax.axvline(x=mode, color='red', ls=':', lw=1.5, label=f'Mode = {mode:.2f}')
    ax.set_title(label, fontsize=10)
    ax.set_xlabel('θ'); ax.set_ylabel('p(θ)')
    ax.legend(fontsize=8)
    ax.set_xlim(0, 1)

plt.tight_layout()
plt.show()

---
### 2.3 Bayesian Update: Beta Prior + Bernoulli → Beta Posterior

$$p(\theta \mid \mathcal{D}) = \text{Beta}(\theta \mid N_1 + a,\ N_0 + b)$$

Exponents simply add — an elegantly simple update rule!

In [ ]:
# ── Coin Flipping: Sequential Bayesian Update ─────────────────────────────────
np.random.seed(7)
theta_true = 0.7   # true heads probability (unknown)
N_total    = 20
flips      = np.random.binomial(1, theta_true, N_total)  # 1=heads 0=tails

a0, b0 = 2, 2      # prior parameters
checkpoints = [0, 1, 3, 5, 10, 20]

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle(f'Beta-Binomial Bayesian Update  (true θ={theta_true}, prior Beta({a0},{b0}))',
             fontsize=13, fontweight='bold')

for ax, n in zip(axes.flat, checkpoints):
    N1 = flips[:n].sum()
    N0 = n - N1
    a_post = a0 + N1
    b_post = b0 + N0

    prior_pdf   = beta_dist.pdf(theta_plot, a0, b0)
    post_pdf    = beta_dist.pdf(theta_plot, a_post, b_post)

    if n > 0:
        # [H4] FIXED: unnecessary N1_arr variable removed
        # [H1] FIXED: np.trapz → np.trapezoid (NumPy 2.0 compatible)
        lik = theta_plot ** N1 * (1 - theta_plot) ** N0
        lik /= np.trapezoid(lik, theta_plot)  # normalize
        ax.plot(theta_plot, lik, 'g-', lw=1.5, alpha=0.7, label='Likelihood')

    ax.plot(theta_plot, prior_pdf, '--', color='gray', lw=1.5, alpha=0.7, label=f'Prior Beta({a0},{b0})')
    ax.plot(theta_plot, post_pdf, color=BLUE, lw=2.5, label=f'Posterior Beta({a_post},{b_post})')
    ax.axvline(x=theta_true, color='red', ls=':', lw=2, alpha=0.8, label=f'True θ={theta_true}')

    mle  = N1/n if n > 0 else 0.5
    map_ = (a_post - 1)/(a_post + b_post - 2) if (a_post > 1 and b_post > 1) else 0.5
    mean = a_post / (a_post + b_post)

    info = f'N={n}, H={N1}, T={N0}'
    if n > 0:
        info += f'\nMLE={mle:.2f}, MAP={map_:.2f}, Mean={mean:.2f}'
    ax.set_title(info, fontsize=10)
    ax.set_xlabel('θ'); ax.set_ylabel('Density')
    ax.legend(fontsize=7)
    ax.set_xlim(0, 1)

plt.tight_layout()
plt.show()

---
### 2.4 MLE, MAP and Posterior Mean — Formulas and Comparison

$$\hat{\theta}_{MLE} = \frac{N_1}{N}, \qquad
\hat{\theta}_{MAP} = \frac{N_1 + a - 1}{N + a + b - 2}, \qquad
\bar{\theta} = \frac{N_1 + a}{N + a + b}$$

Posterior mean = prior-weighted MLE:
$$\mathbb{E}[\theta\mid\mathcal{D}] = \lambda m_1 + (1-\lambda)\hat{\theta}_{MLE}, \qquad \lambda=\frac{\alpha_0}{N+\alpha_0}$$

In [ ]:
# ── MLE / MAP / Posterior Mean: convergence vs N ──────────────────────────────
theta_true_est = 0.6
a, b = 3, 3          # Beta(3,3) prior → prior mean = 0.5
alpha0 = a + b       # effective sample size
m1     = a / alpha0  # prior mean

N_range = np.arange(1, 201)
np.random.seed(0)
all_flips = np.random.binomial(1, theta_true_est, 200)

mle_vals, map_vals, mean_vals, lambda_vals = [], [], [], []
for n in N_range:
    N1 = all_flips[:n].sum()
    N0 = n - N1
    mle_vals.append(N1 / n)
    ap, bp = a + N1, b + N0
    map_vals.append((ap - 1) / (ap + bp - 2) if ap > 1 and bp > 1 else 0.5)
    mean_vals.append(ap / (ap + bp))
    lambda_vals.append(alpha0 / (n + alpha0))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('MLE · MAP · Posterior Mean — Convergence as N Grows', fontsize=13, fontweight='bold')

ax = axes[0]
ax.plot(N_range, mle_vals,  '#E53935', lw=2, label='MLE')
ax.plot(N_range, map_vals,  '#FB8C00', lw=2, label='MAP')
ax.plot(N_range, mean_vals, '#1565c0', lw=2, label='Posterior Mean')
ax.axhline(y=theta_true_est, color='black', ls='--', lw=1.5, alpha=0.7, label=f'True θ={theta_true_est}')
ax.axhline(y=m1, color='gray', ls=':', lw=1.5, label=f'Prior mean m₁={m1:.2f}')
ax.set_xlabel('N'); ax.set_ylabel('θ estimate')
ax.set_title(f'Beta({a},{b}) prior, true θ={theta_true_est}')
ax.legend(fontsize=9)

ax = axes[1]
ax.plot(N_range, lambda_vals, BLUE, lw=2.5)
ax.fill_between(N_range, lambda_vals, alpha=0.2, color=BLUE)
ax.set_xlabel('N'); ax.set_ylabel('λ = α₀ / (N + α₀)')
ax.set_title('Prior Weight λ: prior shrinks as N grows\nE[θ|D] = λ·m₁ + (1-λ)·θ̂_MLE')
ax.axhline(y=0.5, ls='--', color='gray', alpha=0.5)
ax.annotate(f'α₀ = {alpha0}', xy=(alpha0, 0.5), xytext=(alpha0+10, 0.55),
            arrowprops=dict(arrowstyle='->', color='red'), fontsize=11, color='red')

plt.tight_layout()
plt.show()

print('\n📊 Detail for N=5:')
n = 5; N1 = all_flips[:n].sum(); N0 = n - N1
ap, bp = a + N1, b + N0
lam = alpha0 / (n + alpha0)
mle = N1/n; mean = ap/(ap+bp)
print(f'   N₁={N1}, N₀={N0}')
print(f'   MLE = {mle:.3f}')
print(f'   Posterior mean = λ·m₁ + (1-λ)·MLE = {lam:.2f}·{m1:.2f} + {1-lam:.2f}·{mle:.2f} = {mean:.3f}')

---
### 2.5 Zero-Count Problem & Laplace Succession Rule

**Problem:** N=3 tails → MLE: θ̂ = 0 → probability of heads is estimated as zero!

**Solution (Laplace, Beta(1,1) prior):**
$$p(\tilde{x}=1 \mid \mathcal{D}) = \frac{N_1 + 1}{N_1 + N_0 + 2}$$

In [ ]:
# ── Sıfır Sayım Problemi ───────────────────────────────────────────────────────
print('🦢 ZERO-COUNT PROBLEM — Black Swan Paradox')
print('='*58)

experiments = [
    (0, 3,  'N=3 tails (0 heads)'),
    (0, 10, 'N=10 tails (0 heads)'),
    (3, 7,  'N=10, 3 heads 7 tails'),
    (7, 3,  'N=10, 7 heads 3 tails'),
]

for N1, N0, desc in experiments:
    N = N1 + N0
    mle       = N1 / N if N > 0 else 0
    laplace   = (N1 + 1) / (N + 2)        # Beta(1,1) prior
    bayes_22  = (N1 + 2) / (N + 4)        # Beta(2,2) prior
    danger    = '  ⚠️ Dangerous!' if mle == 0 else ''
    print(f'\n  {desc}')
    print(f'    MLE:             p(heads) = {mle:.4f}{danger}')
    print(f'    Laplace:         p(heads) = {laplace:.4f}')
    print(f'    Bayes Beta(2,2): p(heads) = {bayes_22:.4f}')

# Visualisation: effect of different prior choices when N₁=0
N0_vals = np.arange(1, 21)
fig, ax = plt.subplots(figsize=(10, 5))

for (a_lap, b_lap, label, color) in [
        (0, 0, 'MLE (a=0,b=0)',         '#E53935'),
        (1, 1, 'Laplace / Add-1 (a=b=1)','#1565c0'),
        (2, 2, 'Beta(2,2)',              '#2E7D32'),
        (5, 5, 'Beta(5,5) strong prior', '#7B1FA2'),
]:
    preds = [(0 + a_lap) / (n0 + a_lap + b_lap) if (n0 + a_lap + b_lap) > 0 else 0
             for n0 in N0_vals]
    ax.plot(N0_vals, preds, 'o-', lw=2, ms=6, color=color, label=label)

ax.set_xlabel('N₀ (consecutive tails, N₁=0)', fontsize=12)
ax.set_ylabel('p(x̃=1 | D) — heads prediction', fontsize=12)
ax.set_title('Zero-Count: MLE collapses to zero, Bayesian methods correct it', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(-0.02, 0.55)
plt.tight_layout()
plt.show()

---
### 2.6 Posterior Variance and Credible Interval

$$\text{var}[\theta \mid \mathcal{D}] \approx \frac{\hat{\theta}(1-\hat{\theta})}{N}, \qquad
\sigma \approx \sqrt{\frac{\hat{\theta}(1-\hat{\theta})}{N}}$$

In [ ]:
# ── Posterior Variance and Credible Bands ──────────────────────────────────────
theta_true_var = 0.65
a_v, b_v = 2, 2
N_range_v = np.arange(5, 201, 5)
np.random.seed(42)
flips_v = np.random.binomial(1, theta_true_var, 200)

means_v, stds_v, exact_stds = [], [], []
for n in N_range_v:
    N1v = flips_v[:n].sum(); N0v = n - N1v
    ap = a_v + N1v; bp = b_v + N0v
    mean_v = ap / (ap + bp)
    # Exact Beta variance
    var_exact = (ap * bp) / ((ap + bp)**2 * (ap + bp + 1))
    # Approximate variance (N >> a,b)
    theta_hat = N1v / n
    var_approx = theta_hat * (1 - theta_hat) / n
    means_v.append(mean_v)
    stds_v.append(np.sqrt(var_approx))
    exact_stds.append(np.sqrt(var_exact))

means_v = np.array(means_v)
stds_v  = np.array(stds_v)
exact_stds = np.array(exact_stds)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Posterior Variance and Uncertainty Reduction', fontsize=13, fontweight='bold')

ax = axes[0]
ax.plot(N_range_v, means_v, BLUE, lw=2.5, label='Posterior Mean')
ax.fill_between(N_range_v, means_v - 2*exact_stds, means_v + 2*exact_stds,
                alpha=0.2, color=BLUE, label='±2σ (exact variance)')
ax.fill_between(N_range_v, means_v - stds_v, means_v + stds_v,
                alpha=0.35, color='orange', label='±1σ (approximate)')
ax.axhline(y=theta_true_var, color='red', ls='--', lw=2, label=f'True θ={theta_true_var}')
ax.set_xlabel('N'); ax.set_ylabel('θ estimate')
ax.set_title('Credible interval narrows as N increases')
ax.legend(fontsize=9)

ax = axes[1]
ax.plot(N_range_v, exact_stds, BLUE, lw=2.5, label='Exact σ (Beta)')
ax.plot(N_range_v, stds_v,    'orange', lw=2, ls='--', label='Approximate σ = √(θ̂(1-θ̂)/N)')
N_theory = np.linspace(5, 200, 300)
ax.plot(N_theory, np.sqrt(0.25 / N_theory), 'gray', ls=':', lw=1.5, label='Upper bound: 0.5/√N')
ax.set_xlabel('N'); ax.set_ylabel('Posterior Standard Deviation σ')
ax.set_title('Uncertainty: σ ~ 1/√N rate of decrease')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print('\n💡 Key Point: Halving uncertainty requires 4× more data (σ ∝ 1/√N).')

---
### 2.7 Compound Beta-Binomial Distribution — Future Prediction

The probability of x heads in M future trials:

$$\text{Bb}(x \mid a, b, M) = \binom{M}{x}\frac{B(x+a,\,M-x+b)}{B(a,b)}$$

$$\mathbb{E}[x] = M\frac{a}{a+b}, \qquad \text{var}[x] = M\frac{ab}{(a+b)^2}\frac{a+b+M}{a+b+1}$$

In [ ]:
# ── Compound Beta-Binomial ─────────────────────────────────────────────────────────
def beta_binom_pmf(x, a, b, M):
    """Bb(x|a,b,M) PMF — numerical stability via log-space."""
    log_pmf = (gammaln(M + 1) - gammaln(x + 1) - gammaln(M - x + 1)
               + betaln(x + a, M - x + b) - betaln(a, b))
    return np.exp(log_pmf)

M = 20   # number of future trials

prior_configs = [
    (1,  1,  '#78909C', 'Beta(1,1) — Laplace (uniform)'),
    (3,  7,  '#EF5350', 'Beta(3,7) — Tails bias'),
    (7,  3,  '#42A5F5', 'Beta(7,3) — Heads bias'),
    (10, 10, '#66BB6A', 'Beta(10,10) — Balanced strong'),
]

# Observed data: 6 heads, 4 tails
N1_obs, N0_obs = 6, 4

x_vals = np.arange(0, M + 1)
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle(f'Compound Beta-Binomial: Prediction of Heads Count in M={M} Future Trials\n'
             f'(Observation: N₁={N1_obs} heads, N₀={N0_obs} tails)',
             fontsize=13, fontweight='bold')

for ax, (a, b, color, label) in zip(axes.flat, prior_configs):
    # Posterior parameters
    a_post = a + N1_obs
    b_post = b + N0_obs

    pmf_bb    = np.array([beta_binom_pmf(x, a_post, b_post, M) for x in x_vals])
    pmf_binom = binom.pmf(x_vals, M, N1_obs / (N1_obs + N0_obs))  # MLE plug-in

    E_x   = M * a_post / (a_post + b_post)
    var_x = M * a_post * b_post / (a_post + b_post)**2 * (a_post + b_post + M) / (a_post + b_post + 1)

    ax.bar(x_vals - 0.2, pmf_bb,    0.4, color=color, alpha=0.8, label='Beta-Binom (Bayesian)')
    ax.bar(x_vals + 0.2, pmf_binom, 0.4, color='gray', alpha=0.6, label='Binom/MLE (Plug-In)')
    ax.axvline(x=E_x, color='red', ls='--', lw=1.8, label=f'E[x]={E_x:.1f}')
    ax.set_title(f'{label}\nSonsal: Beta({a_post},{b_post})  —  E[x]={E_x:.1f}, σ={np.sqrt(var_x):.1f}',
                 fontsize=10)
    ax.set_xlabel('x (head count)'); ax.set_ylabel('Probability')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

print("\n💡 Beta-Binomial vs. Binom/MLE:")
print('   → Wider (contains more uncertainty)')
print('   → Accounts for prior knowledge')
print('   → More reliable predictions with little data')

---
## 📌 SECTION 3: Multi-Category Generalisation — Dirichlet-Multinomial
### 3.1 Multinomial Likelihood and Dirichlet Prior

K kategorili model için:
$$p(\mathcal{D}\mid\boldsymbol{\theta}) = \prod_{k=1}^K \theta_k^{N_k}$$

$$\text{Dir}(\boldsymbol{\theta}\mid\boldsymbol{\alpha}) = \frac{1}{B(\boldsymbol{\alpha})}\prod_{k=1}^K \theta_k^{\alpha_k - 1}$$

In [ ]:
# ── Dirichlet Distribution: K=3 Simplex Visualisation ─────────────────────────
def sample_dirichlet_simplex(alpha, n_samples=5000):
    """Dirichlet örneklerini eşkenar üçgen üzerine barycentric koordinatla projekte eder.
    Köşeler: A=(0,0): θ₁=1, B=(1,0): θ₂=1, C=(0.5, √3/2): θ₃=1
    """
    samples = np.random.dirichlet(alpha, n_samples)
    # [H8] FIXED: samples.sum(1) is always 1.0 — unnecessary division removed
    # Doğru barycentric koordinatlar:
    x = samples[:, 1] + samples[:, 2] * 0.5
    y = samples[:, 2] * (np.sqrt(3) / 2)
    return x, y

alpha_configs = [
    ([1,   1,   1  ], 'Dir(1,1,1) — Uniform (Laplace)'),
    ([5,   5,   5  ], 'Dir(5,5,5) — Balanced dense'),
    ([0.5, 0.5, 0.5], 'Dir(0.5,0.5,0.5) — Corner-concentrated (sparse)'),
    ([5,   2,   1  ], 'Dir(5,2,1) — θ₁ dominant'),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('Dirichlet Distribution — Probability Simplex for K=3', fontsize=14, fontweight='bold')

for ax, (alpha, title) in zip(axes.flat, alpha_configs):
    np.random.seed(0)
    x, y = sample_dirichlet_simplex(alpha)
    ax.scatter(x, y, s=3, alpha=0.3, color=BLUE)
    # Triangle boundary
    tri_x = [0, 1, 0.5, 0]; tri_y = [0, 0, np.sqrt(3)/2, 0]
    ax.plot(tri_x, tri_y, 'k-', lw=1.5)
    ax.text(-0.05, -0.05, 'θ₁=1', fontsize=10, ha='center')
    ax.text(1.05,  -0.05, 'θ₂=1', fontsize=10, ha='center')
    ax.text(0.5,  np.sqrt(3)/2 + 0.04, 'θ₃=1', fontsize=10, ha='center')
    ax.set_title(title, fontsize=11)
    ax.set_aspect('equal'); ax.axis('off')

plt.tight_layout()
plt.show()

---
### 3.2 Dirichlet-Multinomial Update

$$p(\boldsymbol{\theta}\mid\mathcal{D}) = \text{Dir}(\boldsymbol{\theta}\mid\alpha_1+N_1,\ldots,\alpha_K+N_K)$$

**Posterior Prediction (single trial):**
$$p(X=j\mid\mathcal{D}) = \frac{\alpha_j + N_j}{\alpha_0 + N}$$

In [ ]:
# ── Die Rolling: Dirichlet-Multinomial ─────────────────────────────────────────
np.random.seed(42)
K = 6   # A fair die
theta_true_die = np.array([0.1, 0.1, 0.1, 0.2, 0.2, 0.3])  # loaded die
theta_true_die /= theta_true_die.sum()

alpha_prior = np.ones(K)   # Dir(1,...,1): uniform prior
alpha0 = alpha_prior.sum()

N_total_die = 100
rolls = np.random.choice(K, size=N_total_die, p=theta_true_die)
counts = np.bincount(rolls, minlength=K)

# Bayesçi güncelleme adımları
checkpoints_die = [0, 5, 20, 50, 100]
fig, axes = plt.subplots(1, len(checkpoints_die), figsize=(16, 5), sharey=True)
fig.suptitle('Dirichlet-Multinomial: Learning Die Probabilities (Loaded Die)',
             fontsize=13, fontweight='bold')

labels = [f'Face {i+1}' for i in range(K)]

for ax, n in zip(axes, checkpoints_die):
    cnt = np.bincount(rolls[:n], minlength=K)
    alpha_post = alpha_prior + cnt
    N_total_n  = cnt.sum()

    # MLE (or uniform prior case)
    mle_est = cnt / N_total_n if N_total_n > 0 else np.ones(K)/K
    # Posterior mean
    post_mean = alpha_post / alpha_post.sum()

    x_pos = np.arange(K)
    ax.bar(x_pos - 0.2, theta_true_die, 0.2, color='black', alpha=0.4, label='True θ')
    ax.bar(x_pos,       mle_est,        0.2, color='orange', alpha=0.8, label='MLE')
    ax.bar(x_pos + 0.2, post_mean,      0.2, color=BLUE,    alpha=0.8, label='Posterior mean')

    ax.set_xticks(x_pos)
    ax.set_xticklabels(labels, rotation=45, fontsize=8)
    ax.set_title(f'N = {n}', fontsize=11)
    ax.set_ylabel('Probability')
    if n == checkpoints_die[0]:
        ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

# Numerical summary
print('\n📊 Estimation Summary (N=100):')
alpha_post_final = alpha_prior + counts
post_mean_final  = alpha_post_final / alpha_post_final.sum()
mle_final        = counts / counts.sum()

# [H2] FIXED: f-string outer delimiter changed to double quote
print(f"  {'Face':>6}  {'True':>8}  {'MLE':>8}  {'Post. Mean':>12}  {'Err(MLE)':>10}  {'Err(Bayes)':>12}")
print('-'*65)
for i in range(K):
    print(f'  {i+1:>6}  {theta_true_die[i]:>8.4f}  {mle_final[i]:>8.4f}  '
          f'{post_mean_final[i]:>12.4f}  '
          f'{abs(mle_final[i]-theta_true_die[i]):>10.4f}  '
          f'{abs(post_mean_final[i]-theta_true_die[i]):>12.4f}')

---
### 3.3 Add-K Smoothing — Dirichlet Laplace

β hiper-parametresiyle **add-k düzeltmesi:**
$$p(X=j\mid\mathcal{D}) = \frac{N_j + \beta}{N + K\beta}$$

- β = 1: Laplace (add-one)
- β → 0: MLE
- β → ∞: Uniform tahmin

In [ ]:
# ── Add-K Smoothing: Text Classification Simulation ──────────────────────────
# A small word-count scenario
vocab = ['ai', 'learning', 'model', 'data', 'training', 'binom', 'rare_word']
K_vocab = len(vocab)

# True word frequencies
true_freq = np.array([0.30, 0.25, 0.20, 0.15, 0.05, 0.04, 0.01])

# Small observation set (some words never seen!)
observed_counts = np.array([15, 12, 9, 7, 2, 0, 0])
N_obs = observed_counts.sum()

beta_vals = [0.0001, 0.5, 1.0, 5.0]
labels_b  = ['MLE (β≈0)', 'Add-0.5', 'Laplace (β=1)', 'Strong prior (β=5)']
colors_b  = ['#E53935', '#FB8C00', '#1565c0', '#388E3C']

fig, ax = plt.subplots(figsize=(12, 6))
x_vocab = np.arange(K_vocab)
width   = 0.15

ax.bar(x_vocab - 1.5*width, true_freq, width, color='black', alpha=0.5, label='True distribution')
for i, (beta, label, color) in enumerate(zip(beta_vals, labels_b, colors_b)):
    preds = (observed_counts + beta) / (N_obs + K_vocab * beta)
    ax.bar(x_vocab + (i - 0.5)*width, preds, width, color=color, alpha=0.8, label=label)

ax.set_xticks(x_vocab)
ax.set_xticklabels(vocab, rotation=30, ha='right', fontsize=11)
ax.set_ylabel('Probability Estimate')
ax.set_title('Add-K Smoothing: Probability Estimates for Rare Words\n'
             '("nadir_kelime" ve "binom" gözlemlenmedi — MLE = 0!)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.set_ylim(0, 0.40)
plt.tight_layout()
plt.show()

print('\n📊 Add-K Results (for rare_word):')
# [H2] FIXED: f-string outer delimiter changed to double quote
print(f"  {'Method':20}  {'p(rare_word)'}")
print('-' * 40)
for beta, label in zip(beta_vals, labels_b):
    p = (0 + beta) / (N_obs + K_vocab * beta)
    warning = '  ← ZERO!' if beta < 0.001 else ''
    print(f'  {label:20}  {p:.6f}{warning}')

---
## 📌 SECTION 4: Mixture Model — Modelling Human Intuition

$$p(h) = \pi_0\, p_{rules}(h) + (1-\pi_0)\, p_{interval}(h)$$

π₀ → how much of a "mathematician" (rule-based) vs. "statistician" (interval-based) is the model?

In [ ]:
# ── Effect of π₀ on Posterior Predictive Distribution ───────────────────────────
data_mix = [16, 8, 2, 64]
pi0_values = [0.1, 0.5, 0.9, 0.99]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle(f'Mixture Prior: Effect of the π₀ Parameter  (D = {data_mix})\n'
             'π₀ büyüdükçe model "matematiksel kurallara" daha çok güvenir',
             fontsize=13, fontweight='bold')

for ax, pi0 in zip(axes.flat, pi0_values):
    prior_mix = build_prior(hypotheses, pi0=pi0)
    post_mix  = compute_posterior(data_mix, hypotheses, prior_mix)
    preds_mix = np.array([posterior_predictive(x, post_mix, hypotheses) for x in xs])
    if preds_mix.max() > 0:
        preds_mix /= preds_mix.max()

    # Find TOP-3 hypotheses
    top3 = sorted(post_mix.items(), key=lambda x: x[1], reverse=True)[:3]

    ax.bar(xs, preds_mix, color=BLUE, alpha=0.8, width=0.8)
    for d in data_mix:
        ax.axvline(x=d, color='red', lw=1.5, alpha=0.8)
    ax.set_title(f'π₀ = {pi0}  (rules weight)\n'
                 + '  '.join([f'{h[:12]}:{p:.2f}' for h,p in top3]),
                 fontsize=10)
    ax.set_xlim(0, 101); ax.set_ylim(0, 1.2)
    ax.set_xticks(range(0, 101, 10))
    ax.set_ylabel('p(x̃ ∈ C|D)')
    ax.set_xlabel('x̃')

plt.tight_layout()
plt.show()

print('\n💡 Observation:')
print("  π₀ large (0.9-0.99): Model leans toward sharp rules like 'powers of 2'")
print('  π₀ small (0.1):      Model gives more weight to interval hypotheses')
print('  Humans are typically modelled with π₀ > 0.5 (rule-based thinking dominant)')

---
## 📌 SECTION 5: Comprehensive Comparison — MLE vs MAP vs Bayes


In [ ]:
# ── 5.1 Summary Comparison of Three Methods ─────────────────────────────────────
np.random.seed(1)
theta_compare = 0.7
a_c, b_c = 2, 5   # Tails-biased prior (around θ=0.5)

n_experiments = [3, 5, 10, 20, 50, 100, 500]
results = {'MLE': [], 'MAP': [], 'Post. Mean': [], 'N': n_experiments}

all_flips_cmp = np.random.binomial(1, theta_compare, 500)

for n in n_experiments:
    N1c = all_flips_cmp[:n].sum()
    N0c = n - N1c
    ap, bp = a_c + N1c, b_c + N0c

    results['MLE'].append(N1c / n)
    results['MAP'].append((ap - 1) / (ap + bp - 2))
    results['Post. Mean'].append(ap / (ap + bp))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'MLE vs MAP vs Bayesian Mean  (true θ={theta_compare}, prior Beta({a_c},{b_c}))',
             fontsize=13, fontweight='bold')

ax = axes[0]
for method, color, ls in [('MLE','#E53935','-'), ('MAP','#FB8C00','--'), ('Post. Mean',BLUE,'-')]:
    ax.plot(n_experiments, results[method], color=color, lw=2.5, ls=ls,
            marker='o', ms=6, label=method)
ax.axhline(y=theta_compare, color='black', ls=':', lw=1.5, label=f'True θ={theta_compare}')
ax.axhline(y=a_c/(a_c+b_c), color='gray', ls='-.', lw=1.5, label=f'Prior mean={a_c/(a_c+b_c):.2f}')
ax.set_xscale('log'); ax.set_xlabel('N (log scale)'); ax.set_ylabel('θ estimate')
ax.set_title('θ Estimates vs N'); ax.legend(fontsize=9)

# Error comparison
ax = axes[1]
for method, color, ls in [('MLE','#E53935','-'), ('MAP','#FB8C00','--'), ('Post. Mean',BLUE,'-')]:
    errors = [abs(v - theta_compare) for v in results[method]]
    ax.plot(n_experiments, errors, color=color, lw=2.5, ls=ls,
            marker='s', ms=6, label=f'{method} error')
ax.set_xscale('log'); ax.set_xlabel('N (log scale)'); ax.set_ylabel('|estimate - true|')
ax.set_title('Absolute Error vs N'); ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

# [H2] FIXED: f-string outer delimiter changed to double quote
print('\n📊 Summary Table:')
print(f"  {'N':>6}  {'MLE':>8}  {'MAP':>8}  {'Bayes':>8}")
print('-' * 36)
for i, n in enumerate(n_experiments):
    print(f"  {n:>6}  {results['MLE'][i]:>8.4f}  {results['MAP'][i]:>8.4f}  {results['Post. Mean'][i]:>8.4f}")
print(f'  {"Gerçek":>6}  {theta_compare:>8.4f}  {theta_compare:>8.4f}  {theta_compare:>8.4f}')

In [ ]:
# ── 5.2 Lecture Summary: Key Formulas ────────────────────────────────────────
print('=' * 65)
print('  📚 LECTURE 2 SUMMARY — Key Formulas and Concepts')
print('=' * 65)

sections = [
    ('Number Game / Concept Learning', [
        ('Strong Sampling (Likelihood)', 'p(D|h) = (1/|h|)^N'),
        ('Posterior',                     'p(h|D) ∝ p(D|h)·p(h)'),
        ('BMA',                           'p(x̃∈C|D) = Σ_h p(y=1|x̃,h)·p(h|D)'),
        ('Plug-In',                        'p(x̃∈C|D) ≈ p(x̃|ĥ_MAP)'),
        ('MAP (N→∞)',                      'p(h|D) → δ_{ĥ_MAP}(h)'),
        ('Mixture Prior',                  'p(h) = π₀·p_rules(h) + (1-π₀)·p_interval(h)'),
    ]),
    ('Beta-Binomial (Binary Case)', [
        ('Bernoulli Likelihood',      'p(D|θ) = θ^N₁·(1-θ)^N₀'),
        ('Beta Prior',                 'p(θ) = Beta(θ|a,b)'),
        ('Beta Posterior',             'p(θ|D) = Beta(θ|N₁+a, N₀+b)'),
        ('MLE',                        'θ̂_MLE = N₁/N'),
        ('MAP',                        'θ̂_MAP = (N₁+a-1)/(N+a+b-2)'),
        ('Posterior Mean',             'E[θ|D] = (N₁+a)/(N+a+b)'),
        ('Laplace Succession Rule',    'p(x̃=1|D) = (N₁+1)/(N+2)'),
        ('Compound Beta-Binomial',     'Bb(x|a,b,M) = C(M,x)·B(x+a,M-x+b)/B(a,b)'),
    ]),
    ('Dirichlet-Multinomial (K Categories)', [
        ('Multinomial Likelihood',  'p(D|θ) = ∏_k θ_k^N_k'),
        ('Dirichlet Prior',         'Dir(θ|α) = (1/B(α))·∏_k θ_k^{α_k-1}'),
        ('Dirichlet Posterior',     'p(θ|D) = Dir(θ|α₁+N₁, …, α_K+N_K)'),
        ('Posterior Prediction',    'p(X=j|D) = (α_j+N_j)/(α₀+N)'),
        ('Add-K Smoothing',         'p(X=j|D) = (N_j+β)/(N+K·β)'),
    ]),
]

for sec_title, formulas in sections:
    print(f'\n  ▌ {sec_title}')
    print('  ' + '─'*60)
    for i, (name, formula) in enumerate(formulas, 1):
        print(f'    {i}. {name}')
        print(f'       {formula}')

print('\n' + '=' * 65)
print('  ✅ Notebook complete!')
print('=' * 65)

---
## 🎯 Practice Questions

1. **Number Game:** Compute the posterior predictive distribution when D = {10, 20, 30}. Which hypothesis dominates? Repeat for D = {10, 20, 30, 40, 50}.

2. **Size Principle:** Compute the likelihood ratio p(D|h₁)/p(D|h₂) for |h₁| = 10, |h₂| = 50 at N = 1, 2, 5, 10 observations. At which N does the narrow hypothesis dominate?

3. **Beta-Binomial:** Starting with the prior a=1, b=1 (Laplace), update step-by-step for the sequence HTTHTHHHT. Compare MAP, MLE and posterior mean at each step.

4. **Zero-Count:** No heads observed in N=5 trials. Compute p(x̃=1|D) for β=0, 0.5, 1, 2. Which is most reasonable?

5. **Dirichlet:** Roll a 6-sided die 30 times and update the Dirichlet posterior for K=6. Interpret the posterior probability of the fair die (θ_k = 1/6) hypothesis.

6. **Mixture Prior:** Compare the posterior predictive distributions for π₀ = 0.3 and π₀ = 0.95 given D={16}. Interpret the difference between the two cases.

---
*Generative AI — Haydar Kılıç, Artificial Intelligence Engineering*